<a href="https://colab.research.google.com/github/lukalklikadze/walmart-sales-forecasting/blob/main/notebooks/model_experiment_LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip -q install mlflow kaggle

!git clone -q https://github.com/lukalklikadze/walmart-sales-forecasting.git /content/repo \
    2>/dev/null || (cd /content/repo && git pull -q)
import sys; sys.path.append("/content/repo")

from src.config import setup_env, DATA_DIR, KAGGLE_COMP
tracking_uri = setup_env()
print("MLflow →", tracking_uri)

import os, glob, zipfile, subprocess
os.makedirs(DATA_DIR, exist_ok=True)
subprocess.run(["kaggle","competitions","download","-c",KAGGLE_COMP,"-p",DATA_DIR,"--force"], check=True)
with zipfile.ZipFile(os.path.join(DATA_DIR, KAGGLE_COMP + ".zip")) as z: z.extractall(DATA_DIR)
for inner in glob.glob(os.path.join(DATA_DIR, "*.csv.zip")):
    with zipfile.ZipFile(inner) as z: z.extractall(DATA_DIR)

from src.data import load_raw
train, test, stores, features = load_raw()
print("train:", train.shape, "| test:", test.shape)


MLflow → https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow
train: (421570, 16) | test: (115064, 15)


In [10]:
from src.train_val_split import time_based_split
from src.features import (
    CalendarFeatures,
    LagFeatures,
    DropColumns,
    LAG_CONFIG_BASELINE,
    LAG_CONFIG_SUBMISSION,
)
from src.metrics import wmae
from src.config import DATA_DIR, TRACKING_URI

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
from sklearn.pipeline import Pipeline

mlflow.set_experiment("lightGBM_Training")
print(f"MLflow tracking URI: {tracking_uri}")

MLflow tracking URI: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow


split the data between train and validation

In [12]:
train_split, val_split = time_based_split(train, date_col="Date", val_weeks=39)
VAL_WEEKS = 39

print(f"train_split: {len(train_split):,} rows  {train_split['Date'].min().date()} -> {train_split['Date'].max().date()}")
print(f"val_split:   {len(val_split):,} rows  {val_split['Date'].min().date()} -> {val_split['Date'].max().date()}")

train_split: 305,982 rows  2010-02-05 -> 2012-01-27
val_split:   115,588 rows  2012-02-03 -> 2012-10-26


Feature pipeline

`CalendarFeatures -> LagFeatures(**lag_config) -> DropColumns(["Date"])`.

`LagFeatures.fit` snapshots training history internally, so we `fit_transform` on
`train_split` (passing `y=Weekly_Sales` so the Pipeline forwards it to every step's
`fit`) and only `transform` on `val_split` — this is what keeps validation
leakage-free, since `val_split`'s own sales never enter `LagFeatures.history_`.


In [29]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import TransformedTargetRegressor



LGB_PARAMS = dict(
    objective="regression_l1",   # MAE objective pairs naturally with a WMAE eval metric
    metric="mae",               # Changed from "None" to "mae" to ensure weights are preserved for feval
    boosting_type="gbdt",
    num_leaves=63,
    max_depth=-1,
    learning_rate=0.05,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    min_data_in_leaf=20,
    verbosity=-1,
    seed=42,
)

NUM_BOOST_ROUND = 3000
EARLY_STOPPING_ROUNDS = 100

# log1p that also works for the negative (returns) rows
def signed_log1p(y):
    return np.sign(y) * np.log1p(np.abs(y))

def signed_expm1(z):
    return np.sign(z) * np.expm1(np.abs(z))

CATEGORICAL_COLS = ["Store", "Dept"]

def holiday_weights(is_holiday):
    return np.where(np.asarray(is_holiday) == 1, 5.0, 1.0)


def to_lgb_frame(df, train_categories=None):
    """Split a transformed frame into X (with Store/Dept as category dtype), y, weights."""
    y = df["Weekly_Sales"].to_numpy(dtype=float) if "Weekly_Sales" in df.columns else None
    X = df.drop(columns=["Weekly_Sales"]).copy() if "Weekly_Sales" in df.columns else df.copy()
    w = holiday_weights(df["IsHoliday"])

    for col in CATEGORICAL_COLS:
        if train_categories is None:
            X[col] = X[col].astype("category")
        else:
            X[col] = pd.Categorical(X[col], categories=train_categories[col])

    return X, y, w


class ToLGBFrame(BaseEstimator):
    """sklearn-compatible wrapper around to_lgb_frame, so categorical
    alignment (fitted on train) can live inside a Pipeline step."""
    def fit(self, X, y=None):
        X_lgb, _, _ = to_lgb_frame(X)
        self.train_categories_ = {c: X_lgb[c].cat.categories for c in CATEGORICAL_COLS}
        return self

    def transform(self, X):
        X_lgb, _, _ = to_lgb_frame(X, train_categories=self.train_categories_)
        return X_lgb


class LGBMBoosterRegressor(BaseEstimator, RegressorMixin):
    """sklearn-compatible wrapper around a raw lgb.Booster."""
    def __init__(self, num_boost_round=100, **lgb_params):
        self.num_boost_round = num_boost_round
        self.lgb_params = lgb_params

    def fit(self, X, y, sample_weight=None):
        train_set = lgb.Dataset(
            X, label=y, weight=sample_weight,
            categorical_feature=CATEGORICAL_COLS, free_raw_data=False,
        )
        self.booster_ = lgb.train(self.lgb_params, train_set, num_boost_round=self.num_boost_round)
        return self

    def predict(self, X):
        return self.booster_.predict(X, num_iteration=self.booster_.best_iteration)


def build_full_pipeline(lag_config, log_target, num_boost_round, **lgb_overrides):
    """The Pipeline the assignment wants: raw df in -> predictions out."""
    regressor = LGBMBoosterRegressor(num_boost_round=num_boost_round, **{**LGB_PARAMS, **lgb_overrides})
    if log_target:
        regressor = TransformedTargetRegressor(regressor=regressor, func=signed_log1p, inverse_func=signed_expm1)
    return Pipeline([
        ("calendar", CalendarFeatures()),
        ("lags",     LagFeatures(**lag_config)),
        ("drop",     DropColumns(columns=["Date"])),
        ("to_lgb",   ToLGBFrame()),
        ("model",    regressor),
    ])

 Train + log one config to MLflow

Runs the full fit -> WMAE-early-stopped train -> log cycle for a given lag config,
so we can call it once for `LAG_CONFIG_BASELINE` and once for `LAG_CONFIG_SUBMISSION`
and compare both as separate MLflow runs.


In [14]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
import mlflow.sklearn
import mlflow.lightgbm
from sklearn.compose import TransformedTargetRegressor
import mlflow # Import mlflow for active_run()

# Redefine wmae here to ensure correct signature and implementation
def wmae(preds, lgb_train):
    """
    Weighted Mean Absolute Error for LightGBM.
    Ensures correct signature for LightGBM's feval.
    """
    labels = lgb_train.get_label()
    weights = lgb_train.get_weight()
    # Handle case where get_weight() returns None (e.g., when all weights are 1)
    if weights is None:
        weights = np.ones_like(labels)
    return 'wmae', (np.sum(weights * np.abs(labels - preds)) / np.sum(weights)), False

# Helper function to calculate WMAE on a given scale
def calculate_wmae(y_true, y_pred, weights):
    """
    Calculates WMAE given true values, predictions, and weights.
    """
    if weights is None:
        weights = np.ones_like(y_true)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


# Dictionary to store results for comparison
results = {}

def build_feature_pipeline(lag_config):
    return Pipeline([
        ("calendar", CalendarFeatures()),
        ("lags",     LagFeatures(**lag_config)),
        ("drop",     DropColumns(columns=["Date"])), # Explicitly passing columns here
    ])


def train_and_log(run_name,  lag_config, log_target=False, weighted=False, **lgb_overrides):
      # Build the feature engineering pipeline
      feature_pipeline = build_feature_pipeline(lag_config)

      # Capture original validation target for later comparison
      # `val_split` contains the original 'Weekly_Sales' values before any transformations.
      y_val_original_untransformed = val_split["Weekly_Sales"].copy()


      # Fit the feature pipeline on the training data
      feature_pipeline.fit(train_split, y=train_split["Weekly_Sales"])

      # Transform training and validation data
      train_t = feature_pipeline.transform(train_split)
      val_t = feature_pipeline.transform(val_split)


      # Apply target transformation if log_target is True
      if log_target:
          train_t["Weekly_Sales"] = signed_log1p(train_t["Weekly_Sales"])
          val_t["Weekly_Sales"] = signed_log1p(val_t["Weekly_Sales"])

      X_train, y_train, w_train_computed = to_lgb_frame(train_t)
      train_categories = {c: X_train[c].cat.categories for c in CATEGORICAL_COLS}
      # y_val_transformed will hold the potentially log-transformed target for validation set
      X_val, y_val_transformed, w_val_computed = to_lgb_frame(val_t, train_categories=train_categories)

      # Conditionally apply weights
      actual_w_train = w_train_computed if weighted else np.ones_like(y_train)
      actual_w_val = w_val_computed if weighted else np.ones_like(y_val_transformed)

      train_set = lgb.Dataset(
          X_train, label=y_train, weight=actual_w_train,
          categorical_feature=CATEGORICAL_COLS, free_raw_data=False,
      )
      val_set = lgb.Dataset(
          X_val, label=y_val_transformed, weight=actual_w_val, reference=train_set,
          categorical_feature=CATEGORICAL_COLS, free_raw_data=False,
      )
          # Log preprocessing related info
          # Log the fitted feature engineering pipeline
          # mlflow.sklearn.log_model(pipeline, artifact_path="feature_pipeline", serialization_format="cloudpickle"
      # Nested run for Training
      with mlflow.start_run(run_name=run_name) as run:
          mlflow.log_params({**LGB_PARAMS, **lgb_overrides}) # Log LightGBM hyperparameters

          booster = lgb.train(
              LGB_PARAMS, # Use LGB_PARAMS directly
              train_set,
              num_boost_round=NUM_BOOST_ROUND,
              valid_sets=[train_set, val_set],
              valid_names=["train", "val"],
              feval=wmae,
              callbacks=[
                  lgb.early_stopping(EARLY_STOPPING_ROUNDS),
                  lgb.log_evaluation(100),
              ],
          )

          mlflow.log_params({"lags": str(lag_config["lags"]), "windows": str(lag_config["windows"]),
                           "log_target": log_target, "weighted": weighted, "val_weeks": VAL_WEEKS})

          # Log WMAE on the scale it was trained (transformed if log_target=True)
          train_wmae_transformed_scale = booster.best_score["train"]["wmae"]
          mlflow.log_metric("train_wmae_transformed_scale", train_wmae_transformed_scale)

          val_wmae_transformed_scale = booster.best_score["val"]["wmae"]
          mlflow.log_metric("val_wmae_transformed_scale", val_wmae_transformed_scale)
          mlflow.log_metric("best_iteration", booster.best_iteration)


          # Calculate WMAE on the original scale for fair comparison
          train_preds = booster.predict(X_train, num_iteration=booster.best_iteration)
          if log_target:
              train_preds_original_scale = signed_expm1(train_preds)
              y_train_original_untransformed = signed_expm1(y_train) # Need to inverse transform y_train for original scale comparison
          else:
              train_preds_original_scale = train_preds
              y_train_original_untransformed = y_train

          train_wmae_original_scale = calculate_wmae(y_train_original_untransformed, train_preds_original_scale, actual_w_train)
          mlflow.log_metric("train_wmae_original_scale", train_wmae_original_scale)


          val_preds = booster.predict(X_val, num_iteration=booster.best_iteration)
          if log_target:
              val_preds_original_scale = signed_expm1(val_preds)
          else:
              val_preds_original_scale = val_preds

          # Ensure `actual_w_val` is used for WMAE calculation on original scale
          val_wmae_original_scale = calculate_wmae(y_val_original_untransformed, val_preds_original_scale, actual_w_val)
          mlflow.log_metric("val_wmae_original_scale", val_wmae_original_scale)


          ax = lgb.plot_importance(booster, max_num_features=20, importance_type="gain")
          fig = ax.figure
          fig.tight_layout()
          fig_path = f"feature_importance_{run_name}.png"
          fig.savefig(fig_path, bbox_inches="tight")
          plt.close(fig)
          mlflow.log_artifact(fig_path)

          # Update results dictionary to store the WMAE on the original scale
          results[run_name] = (val_wmae_original_scale, dict(lag_cfg=lag_config, log_target=log_target,
                                         weighted=weighted, overrides=lgb_overrides))

          # Log the trained LightGBM model
          # mlflow.lightgbm.log_model(booster, artifact_path="lgbm_model") # Changed to log booster, not pipeline

          print(f"[{run_name}] best_iteration={booster.best_iteration}  train_wmae_original_scale={train_wmae_original_scale:.2f}  val_wmae_original_scale={val_wmae_original_scale:.2f}")

      return booster, val_wmae_original_scale, feature_pipeline, run.info.run_id # Return the original scale WMAE

 Train


In [15]:
print("\n--- Running for lag_submission_log1p ---")
_, val_wmae_submission, _, run_id_submission = train_and_log(run_name="lag_submission_log1p", lag_config=LAG_CONFIG_SUBMISSION,
                                                             log_target = True)



--- Running for lag_submission_log1p ---


/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.339902	train's wmae: 0.339902	val's l1: 0.295156	val's wmae: 0.295156
[200]	train's l1: 0.290473	train's wmae: 0.290473	val's l1: 0.276385	val's wmae: 0.276385
[300]	train's l1: 0.263126	train's wmae: 0.263126	val's l1: 0.26958	val's wmae: 0.26958
[400]	train's l1: 0.245761	train's wmae: 0.245761	val's l1: 0.266113	val's wmae: 0.266113
[500]	train's l1: 0.234192	train's wmae: 0.234192	val's l1: 0.264553	val's wmae: 0.264553
[600]	train's l1: 0.224073	train's wmae: 0.224073	val's l1: 0.263306	val's wmae: 0.263306
[700]	train's l1: 0.21752	train's wmae: 0.21752	val's l1: 0.262365	val's wmae: 0.262365
[800]	train's l1: 0.21108	train's wmae: 0.21108	val's l1: 0.261848	val's wmae: 0.261848
[900]	train's l1: 0.20653	train's wmae: 0.20653	val's l1: 0.261564	val's wmae: 0.261564
[1000]	train's l1: 0.202858	train's wmae: 0.202858	val's l1: 0.261354	val's wmae: 0.261354
[1100]	train's l1: 0.199674	train's wmae: 0.1

In [17]:
mlflow.set_tracking_uri(TRACKING_URI)



# Train and log for LAG_CONFIG_BASELINE
print("\n--- Running for lag_baseline ---")
_, val_wmae_baseline, _, run_id_baseline = train_and_log(run_name="lag_baseline", lag_config=LAG_CONFIG_BASELINE)

# Train and log for LAG_CONFIG_SUBMISSION
print("\n--- Running for lag_submission ---")
_, val_wmae_submission, _, run_id_submission = train_and_log(run_name="lag_submission", lag_config=LAG_CONFIG_SUBMISSION)

# print("\n--- Running for lag_submission_log1p ---")
# _, val_wmae_submission, _, run_id_submission = train_and_log(run_name="lag_submission_log1p", lag_config=LAG_CONFIG_SUBMISSION,
#                                                              log_target = True)

print("\n--- Running for lag_submission_weighted ---")
_, val_wmae_submission, _, run_id_submission = train_and_log(run_name="lag_submission_weighted", lag_config=LAG_CONFIG_SUBMISSION,
                                                             weighted = True)

print("\n--- Running for lag_submission_log1p_weighted ---")
_, val_wmae_submission, _, run_id_submission = train_and_log(run_name="lag_submission_log1p_weighted", lag_config=LAG_CONFIG_SUBMISSION,
                                                             log_target = True, weighted = True)



# print("\n--- Comparing Results ---")
# print(f"LAG_CONFIG_BASELINE WMAE: {val_wmae_baseline:.2f}")
# print(f"LAG_CONFIG_SUBMISSION WMAE: {val_wmae_submission:.2f}")

# # Determine the best run based on WMAE (lower is better)
# if val_wmae_baseline <= val_wmae_submission:
#     best_config_name = "LAG_CONFIG_BASELINE"
#     best_run_id = run_id_baseline
# else:
#     best_config_name = "LAG_CONFIG_SUBMISSION"
#     best_run_id = run_id_submission

# print(f"\nThe best performing configuration is: {best_config_name} (Run ID: {best_run_id})")


--- Running for lag_baseline ---


/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 1736.74	train's wmae: 1736.74	val's l1: 7279.98	val's wmae: 7279.98
[200]	train's l1: 1522.94	train's wmae: 1522.94	val's l1: 6305.64	val's wmae: 6305.64
[300]	train's l1: 1436.69	train's wmae: 1436.69	val's l1: 6282.34	val's wmae: 6282.34
[400]	train's l1: 1380.97	train's wmae: 1380.97	val's l1: 6248.11	val's wmae: 6248.11
[500]	train's l1: 1339.1	train's wmae: 1339.1	val's l1: 6248.92	val's wmae: 6248.92
[600]	train's l1: 1307.13	train's wmae: 1307.13	val's l1: 6134.44	val's wmae: 6134.44
[700]	train's l1: 1284.76	train's wmae: 1284.76	val's l1: 6062.24	val's wmae: 6062.24
[800]	train's l1: 1263.59	train's wmae: 1263.59	val's l1: 5997.27	val's wmae: 5997.27
[900]	train's l1: 1244.51	train's wmae: 1244.51	val's l1: 5991.09	val's wmae: 5991.09
Early stopping, best iteration is:
[876]	train's l1: 1250.18	train's wmae: 1250.18	val's l1: 5960.23	val's wmae: 5960.23
[lag_baseline] best_iteration=876  train_wmae

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 2767.01	train's wmae: 2767.01	val's l1: 1831.68	val's wmae: 1831.68
[200]	train's l1: 2408.87	train's wmae: 2408.87	val's l1: 1720.85	val's wmae: 1720.85
[300]	train's l1: 2193.77	train's wmae: 2193.77	val's l1: 1701.17	val's wmae: 1701.17
[400]	train's l1: 2025.04	train's wmae: 2025.04	val's l1: 1691.9	val's wmae: 1691.9
[500]	train's l1: 1912.98	train's wmae: 1912.98	val's l1: 1686.03	val's wmae: 1686.03
[600]	train's l1: 1810.77	train's wmae: 1810.77	val's l1: 1682.34	val's wmae: 1682.34
[700]	train's l1: 1734.51	train's wmae: 1734.51	val's l1: 1682.16	val's wmae: 1682.16
[800]	train's l1: 1665.37	train's wmae: 1665.37	val's l1: 1680.53	val's wmae: 1680.53
[900]	train's l1: 1619.44	train's wmae: 1619.44	val's l1: 1677.05	val's wmae: 1677.05
[1000]	train's l1: 1574.63	train's wmae: 1574.63	val's l1: 1676.94	val's wmae: 1676.94
Early stopping, best iteration is:
[955]	train's l1: 1593.37	train's wmae: 1593

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 2974.96	train's wmae: 2974.96	val's l1: 1826.47	val's wmae: 1826.47
[200]	train's l1: 2611.62	train's wmae: 2611.62	val's l1: 1739.8	val's wmae: 1739.8
[300]	train's l1: 2374.74	train's wmae: 2374.74	val's l1: 1720.61	val's wmae: 1720.61
[400]	train's l1: 2184.05	train's wmae: 2184.05	val's l1: 1712.14	val's wmae: 1712.14
[500]	train's l1: 2065.37	train's wmae: 2065.37	val's l1: 1708.94	val's wmae: 1708.94
[600]	train's l1: 1968.91	train's wmae: 1968.91	val's l1: 1706.39	val's wmae: 1706.39
[700]	train's l1: 1895.05	train's wmae: 1895.05	val's l1: 1704.74	val's wmae: 1704.74
[800]	train's l1: 1833.73	train's wmae: 1833.73	val's l1: 1702.1	val's wmae: 1702.1
[900]	train's l1: 1788.74	train's wmae: 1788.74	val's l1: 1701.07	val's wmae: 1701.07
[1000]	train's l1: 1747.53	train's wmae: 1747.53	val's l1: 1700.98	val's wmae: 1700.98
Early stopping, best iteration is:
[950]	train's l1: 1767.31	train's wmae: 1767.3

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.342909	train's wmae: 0.342909	val's l1: 0.296374	val's wmae: 0.296374
[200]	train's l1: 0.295044	train's wmae: 0.295044	val's l1: 0.281999	val's wmae: 0.281999
[300]	train's l1: 0.269994	train's wmae: 0.269994	val's l1: 0.276517	val's wmae: 0.276517
[400]	train's l1: 0.251867	train's wmae: 0.251867	val's l1: 0.272195	val's wmae: 0.272195
[500]	train's l1: 0.23985	train's wmae: 0.23985	val's l1: 0.270151	val's wmae: 0.270151
[600]	train's l1: 0.229624	train's wmae: 0.229624	val's l1: 0.268759	val's wmae: 0.268759
[700]	train's l1: 0.223067	train's wmae: 0.223067	val's l1: 0.268051	val's wmae: 0.268051
[800]	train's l1: 0.216812	train's wmae: 0.216812	val's l1: 0.267328	val's wmae: 0.267328
[900]	train's l1: 0.21287	train's wmae: 0.21287	val's l1: 0.267133	val's wmae: 0.267133
[1000]	train's l1: 0.209006	train's wmae: 0.209006	val's l1: 0.266903	val's wmae: 0.266903
[1100]	train's l1: 0.20646	train's wmae: 

In [21]:
# pick the best target + weighting combo from the four base runs, then tune around it
base = ["lag_submission", "lag_submission_log1p",
        "lag_submission_weighted", "lag_submission_log1p_weighted"]
best_base = min(base, key=lambda k: results[k][0])
use_log    = results[best_base][1]["log_target"]
use_weight = results[best_base][1]["weighted"]
print(f"tuning around {best_base}: log_target={use_log}, weighted={use_weight}")

LGB_grid = [
   dict(
    num_leaves=31,
    max_depth=6,
    learning_rate=0.05
),
   dict(
    num_leaves=127,
    max_depth=-1,
    learning_rate=0.05
),
   dict(
    num_leaves=127,
    learning_rate=0.02,
    num_boost_round=3000
),
   dict(
    num_leaves=63,
    min_data_in_leaf=100,
    lambda_l2=5
),
   dict(
    num_leaves=63,
    feature_fraction=0.7,
    bagging_fraction=0.7,
    bagging_freq=5
)
]
for g in LGB_grid:
    name = "tuned_" + "_".join(f"{k}{v}" for k, v in g.items())
    _, val_wmae_submission, _, run_id_submission = train_and_log(
        run_name=name,
        lag_config=LAG_CONFIG_SUBMISSION,
        log_target=use_log,
        weighted=use_weight,
        **g,
    )

tuning around lag_submission_log1p: log_target=True, weighted=False


/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.339902	train's wmae: 0.339902	val's l1: 0.295156	val's wmae: 0.295156
[200]	train's l1: 0.290473	train's wmae: 0.290473	val's l1: 0.276385	val's wmae: 0.276385
[300]	train's l1: 0.263126	train's wmae: 0.263126	val's l1: 0.26958	val's wmae: 0.26958
[400]	train's l1: 0.245761	train's wmae: 0.245761	val's l1: 0.266113	val's wmae: 0.266113
[500]	train's l1: 0.234192	train's wmae: 0.234192	val's l1: 0.264553	val's wmae: 0.264553
[600]	train's l1: 0.224073	train's wmae: 0.224073	val's l1: 0.263306	val's wmae: 0.263306
[700]	train's l1: 0.21752	train's wmae: 0.21752	val's l1: 0.262365	val's wmae: 0.262365
[800]	train's l1: 0.21108	train's wmae: 0.21108	val's l1: 0.261848	val's wmae: 0.261848
[900]	train's l1: 0.20653	train's wmae: 0.20653	val's l1: 0.261564	val's wmae: 0.261564
[1000]	train's l1: 0.202858	train's wmae: 0.202858	val's l1: 0.261354	val's wmae: 0.261354
[1100]	train's l1: 0.199674	train's wmae: 0.1

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.339902	train's wmae: 0.339902	val's l1: 0.295156	val's wmae: 0.295156
[200]	train's l1: 0.290473	train's wmae: 0.290473	val's l1: 0.276385	val's wmae: 0.276385
[300]	train's l1: 0.263126	train's wmae: 0.263126	val's l1: 0.26958	val's wmae: 0.26958
[400]	train's l1: 0.245761	train's wmae: 0.245761	val's l1: 0.266113	val's wmae: 0.266113
[500]	train's l1: 0.234192	train's wmae: 0.234192	val's l1: 0.264553	val's wmae: 0.264553
[600]	train's l1: 0.224073	train's wmae: 0.224073	val's l1: 0.263306	val's wmae: 0.263306
[700]	train's l1: 0.21752	train's wmae: 0.21752	val's l1: 0.262365	val's wmae: 0.262365
[800]	train's l1: 0.21108	train's wmae: 0.21108	val's l1: 0.261848	val's wmae: 0.261848
[900]	train's l1: 0.20653	train's wmae: 0.20653	val's l1: 0.261564	val's wmae: 0.261564
[1000]	train's l1: 0.202858	train's wmae: 0.202858	val's l1: 0.261354	val's wmae: 0.261354
[1100]	train's l1: 0.199674	train's wmae: 0.1

...

Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.339902	train's wmae: 0.339902	val's l1: 0.295156	val's wmae: 0.295156
[200]	train's l1: 0.290473	train's wmae: 0.290473	val's l1: 0.276385	val's wmae: 0.276385
[300]	train's l1: 0.263126	train's wmae: 0.263126	val's l1: 0.26958	val's wmae: 0.26958
[400]	train's l1: 0.245761	train's wmae: 0.245761	val's l1: 0.266113	val's wmae: 0.266113
[500]	train's l1: 0.234192	train's wmae: 0.234192	val's l1: 0.264553	val's wmae: 0.264553
[600]	train's l1: 0.224073	train's wmae: 0.224073	val's l1: 0.263306	val's wmae: 0.263306
[700]	train's l1: 0.21752	train's wmae: 0.21752	val's l1: 0.262365	val's wmae: 0.262365
[800]	train's l1: 0.21108	train's wmae: 0.21108	val's l1: 0.261848	val's wmae: 0.261848
[900]	train's l1: 0.20653	train's wmae: 0.20653	val's l1: 0.261564	val's wmae: 0.261564
[1000]	train's l1: 0.202858	train's wmae: 0.202858	val's l1: 0.261354	val's wmae: 0.261354
[1100]	train's l1: 0.199674	train's wmae: 0.1

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.339902	train's wmae: 0.339902	val's l1: 0.295156	val's wmae: 0.295156
[200]	train's l1: 0.290473	train's wmae: 0.290473	val's l1: 0.276385	val's wmae: 0.276385
[300]	train's l1: 0.263126	train's wmae: 0.263126	val's l1: 0.26958	val's wmae: 0.26958
[400]	train's l1: 0.245761	train's wmae: 0.245761	val's l1: 0.266113	val's wmae: 0.266113
[500]	train's l1: 0.234192	train's wmae: 0.234192	val's l1: 0.264553	val's wmae: 0.264553
[600]	train's l1: 0.224073	train's wmae: 0.224073	val's l1: 0.263306	val's wmae: 0.263306
[700]	train's l1: 0.21752	train's wmae: 0.21752	val's l1: 0.262365	val's wmae: 0.262365
[800]	train's l1: 0.21108	train's wmae: 0.21108	val's l1: 0.261848	val's wmae: 0.261848
[900]	train's l1: 0.20653	train's wmae: 0.20653	val's l1: 0.261564	val's wmae: 0.261564
[1000]	train's l1: 0.202858	train's wmae: 0.202858	val's l1: 0.261354	val's wmae: 0.261354
[1100]	train's l1: 0.199674	train's wmae: 0.1

/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.339902	train's wmae: 0.339902	val's l1: 0.295156	val's wmae: 0.295156
[200]	train's l1: 0.290473	train's wmae: 0.290473	val's l1: 0.276385	val's wmae: 0.276385
[300]	train's l1: 0.263126	train's wmae: 0.263126	val's l1: 0.26958	val's wmae: 0.26958
[400]	train's l1: 0.245761	train's wmae: 0.245761	val's l1: 0.266113	val's wmae: 0.266113
[500]	train's l1: 0.234192	train's wmae: 0.234192	val's l1: 0.264553	val's wmae: 0.264553
[600]	train's l1: 0.224073	train's wmae: 0.224073	val's l1: 0.263306	val's wmae: 0.263306
[700]	train's l1: 0.21752	train's wmae: 0.21752	val's l1: 0.262365	val's wmae: 0.262365
[800]	train's l1: 0.21108	train's wmae: 0.21108	val's l1: 0.261848	val's wmae: 0.261848
[900]	train's l1: 0.20653	train's wmae: 0.20653	val's l1: 0.261564	val's wmae: 0.261564
[1000]	train's l1: 0.202858	train's wmae: 0.202858	val's l1: 0.261354	val's wmae: 0.261354
[1100]	train's l1: 0.199674	train's wmae: 0.1

In [22]:

# all runs side by side
board = pd.Series({k: v[0] for k, v in results.items()}).sort_values()
print(board.round(1).to_string())


lag_submission_log1p                                                        1675.2
tuned_num_leaves127_max_depth-1_learning_rate0.05                           1675.2
tuned_num_leaves31_max_depth6_learning_rate0.05                             1675.2
tuned_num_leaves63_feature_fraction0.7_bagging_fraction0.7_bagging_freq5    1675.2
tuned_num_leaves63_min_data_in_leaf100_lambda_l25                           1675.2
tuned_num_leaves127_learning_rate0.02_num_boost_round3000                   1675.2
lag_submission                                                              1676.3
lag_submission_weighted                                                     1700.2
lag_submission_log1p_weighted                                               1766.1
lag_baseline                                                                5960.2


 ## Log the best model

In [30]:
best_name = board.index[0]
_, spec = results[best_name]
client = mlflow.tracking.MlflowClient()
runs = client.search_runs(
    experiment_ids=[mlflow.get_experiment_by_name("lightGBM_Training").experiment_id],
    filter_string=f"tags.mlflow.runName = '{best_name}'"
)
best_iteration = int(runs[0].data.metrics["best_iteration"])
print("refitting on full train:", best_name, spec)

final_pipe = build_full_pipeline(
    spec["lag_cfg"], spec["log_target"],
    num_boost_round=best_iteration, **spec["overrides"],
)

fit_kwargs = {}
if spec["weighted"]:
    fit_kwargs["model__sample_weight"] = holiday_weights(train["IsHoliday"])
    if spec["log_target"]:
        fit_kwargs = {f"model__regressor__sample_weight": fit_kwargs.pop("model__sample_weight")}

final_pipe.fit(train.drop(columns=["Weekly_Sales"]), train["Weekly_Sales"], **fit_kwargs)

preds = final_pipe.predict(test)   # raw, unpreprocessed test frame straight through
assert np.isfinite(preds).all()

with mlflow.start_run(run_name=f"final_full_train__{best_name}"):
    mlflow.log_params({"val_wmae_of_chosen_cfg": results[best_name][0],
                        "chosen_run": best_name, "refit_on": "full_train",
                        "weighted": spec["weighted"], "best_iteration": best_iteration})
    mlflow.sklearn.log_model(final_pipe, "model", serialization_format="cloudpickle")

print("final pipeline logged | sample test preds:", preds[:3].round(1))

refitting on full train: lag_submission_log1p {'lag_cfg': {'lags': (52, 53, 104), 'windows': ()}, 'log_target': True, 'weighted': False, 'overrides': {}}


2026/07/09 17:54:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/09 17:54:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run final_full_train__lag_submission_log1p at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/5/runs/89516e0adeb642e58d95cbb67918ce2e
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/5
final pipeline logged | sample test preds: [36728.6 21975.9 21143. ]
